In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [5]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'I can help, but I need a bit more context: which course are you referring to?\n\nIf you mean a specific class, send me:\n- the course name\n- the platform or school\n- whether it’s already started\n\nIn general, you can usually join if:\n- enrollment is still open\n- the class hasn’t ended\n- there are no prerequisites or seat limits blocking you\n\nIf you want, I can also help you draft a quick message asking the instructor or organizer if you can still join.'

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [8]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join after start enrollment deadline"}', call_id='call_MwA4qTq8pzo8FkRQyENhRtug', name='search', type='function_call', id='fc_05964aabbeeab2b1006a33cbda892c819a9005036d7251e42e', namespace=None, status='completed')]

In [9]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [14]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [15]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course and start learning.\n\nIf you want a certificate, though, you’ll need to submit your project while submissions are still being accepted.'

In [16]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(772, 38)

In [17]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [18]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [19]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [20]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment registration late join FAQ"}


In [21]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"certificate project while accepting submissions join now self-paced live cohort FAQ course submissions open deadline"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, the key is to submit your project while the course is still accepting submissions.

Also, you can start learning right away even if you just discovered it; registration isn’t required to begin following the materials.

If you want, I can also explain how the course workflow works or what you need for the certificate.


In [22]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [23]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama run locally install local setup Ollama"}
iteration #2...
function_call: search {"query":"Ollama local server localhost 11434 ollama serve ollama run llama3 course FAQ"}
iteration #3...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - **macOS**: download the `.pkg` from https://ollama.com/download
   - **Windows**: download the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This downloads the model and opens a local chat interface.

3. **Check that the server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from the Ollama server.

4. **If you’re using Python**
   ```bash
   pip install ollama
   ```
   Then:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   prin

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - **macOS**: download the `.pkg` from https://ollama.com/download\n   - **Windows**: download the `.msi`\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and opens a local chat interface.\n\n3. **Check that the server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from the Ollama server.\n\n4. **If you’re using Python**\n   ```bash\n   pip install ollama\n   ```\n   Then:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restart the server with:\n```bash\nollama serve\n```\nor in a notebook:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &

In [24]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered course can I still join"}
function_call: search {"query":"course enrollment late join discovered course still join"}
iteration #2...
ASSISTANT:
Yes — you can still join the course. The course materials are available, so you can start anytime.

One important note: if you want a certificate, you need to submit your project while submissions are still open, and certificates are only awarded for the live cohort, not self-paced participation.

If you’d like, I can also help you with the weekly workflow or certificate requirements.


'Yes — you can still join the course. The course materials are available, so you can start anytime.\n\nOne important note: if you want a certificate, you need to submit your project while submissions are still open, and certificates are only awarded for the live cohort, not self-paced participation.\n\nIf you’d like, I can also help you with the weekly workflow or certificate requirements.'

In [25]:
agent_loop(instructions, "Saya jumpa course ini, boleh masuk ke?")

iteration #1...
function_call: search {"query":"boleh masuk ke course ini pendaftaran akses siapa boleh join"}
function_call: search {"query":"course ini siapa boleh masuk syarat kemasukan pendaftaran"}
function_call: search {"query":"enrollment access who can join this course prerequisites registration"}
iteration #2...
ASSISTANT:
Ya, boleh join. Dari FAQ course ini:

- **Kalau baru jumpa sekarang, masih boleh masuk**
- **Tak perlu tunggu confirmation email** — pendaftaran sebenarnya hanya untuk ukur minat
- Anda juga boleh **terus mula belajar dan hantar homework** selagi borang/submission masih dibuka
- Kalau nak **certificate**, anda perlu **siapkan projek dan hantar semasa cohort masih berjalan**

Kalau anda nak, saya boleh bantu terangkan juga:
- cara mula ikut course ini,
- syarat untuk dapat certificate,
- atau sama ada anda boleh belajar secara self-paced.


'Ya, boleh join. Dari FAQ course ini:\n\n- **Kalau baru jumpa sekarang, masih boleh masuk**\n- **Tak perlu tunggu confirmation email** — pendaftaran sebenarnya hanya untuk ukur minat\n- Anda juga boleh **terus mula belajar dan hantar homework** selagi borang/submission masih dibuka\n- Kalau nak **certificate**, anda perlu **siapkan projek dan hantar semasa cohort masih berjalan**\n\nKalau anda nak, saya boleh bantu terangkan juga:\n- cara mula ikut course ini,\n- syarat untuk dapat certificate,\n- atau sama ada anda boleh belajar secara self-paced.'

In [29]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform at least 4 searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course late discovered can I join"}
function_call: search {"query":"enrollment join course late student discovered course"}
function_call: search {"query":"course access enrollment registration join"}
function_call: search {"query":"FAQ join the course after it starts"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

A couple of useful details:
- You don’t need a confirmation email to start.
- You can begin learning and submitting homework while the submission form is open.
- If you want a certificate, you’ll need to submit your project while submissions are still being accepted.

If you want, I can also help you figure out how to start the course and what to do first. Are there other areas you’d like to explore?


'Yes — you can still join the course.\n\nA couple of useful details:\n- You don’t need a confirmation email to start.\n- You can begin learning and submitting homework while the submission form is open.\n- If you want a certificate, you’ll need to submit your project while submissions are still being accepted.\n\nIf you want, I can also help you figure out how to start the course and what to do first. Are there other areas you’d like to explore?'

In [30]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen gambit"}
iteration #2...
function_call: search {"query":"queen gambit chess opening"}
function_call: search {"query":"queen's gambit chess opening"}
function_call: search {"query":"what is the queen gambit"}
function_call: search {"query":"queen gambit opening definition"}
iteration #3...
ASSISTANT:
“Queen’s Gambit” is a chess opening.

It starts with:
1. d4 d5  
2. c4

White offers the c-pawn to try to control the center and get a better position. If Black takes the pawn, that’s called the **Queen’s Gambit Accepted**; if Black doesn’t, it’s the **Queen’s Gambit Declined**.

If you meant the Netflix show, that’s named after this chess opening too.

Do you want a simple explanation of how the opening works, or the ideas behind it?


'“Queen’s Gambit” is a chess opening.\n\nIt starts with:\n1. d4 d5  \n2. c4\n\nWhite offers the c-pawn to try to control the center and get a better position. If Black takes the pawn, that’s called the **Queen’s Gambit Accepted**; if Black doesn’t, it’s the **Queen’s Gambit Declined**.\n\nIf you meant the Netflix show, that’s named after this chess opening too.\n\nDo you want a simple explanation of how the opening works, or the ideas behind it?'

In [31]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen’s gambit,” so I can’t answer this from the course materials.

If you meant something else course-related, feel free to rephrase it. Are there other areas you want to explore?


'I couldn’t find any course FAQ entry about “queen’s gambit,” so I can’t answer this from the course materials.\n\nIf you meant something else course-related, feel free to rephrase it. Are there other areas you want to explore?'